In [ ]:
%matplotlib inline

# Social

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn import cluster
from sklearn import metrics
from scipy.spatial.distance import cdist
from yellowbrick.cluster import KElbowVisualizer

feat_social = ['size', 'i', 'we', 'pronoun', 'ppron', 'ipron', 'affect', 'posemo', 
               'negemo', 'work', 'power', 'drives', 'percept', 'negate', 'interrog', 
               'focuspresent', 'auxverb', 'you', 'assent', 'focuspast', 'affiliation', 'social']

df = pd.read_csv('100_submission_measures_binary.csv')

min_size_subs = 3

df_social = df[feat_social]
df_social.drop(df_social[df_social['size'] <= min_size_subs].index, inplace = True)
df_social.drop(['size'], axis=1, inplace=True)
df_social

,i,we,pronoun,ppron,ipron,affect,posemo,negemo,work,power,...,percept,negate,interrog,focuspresent,auxverb,you,assent,focuspast,affiliation,social
0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
1,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
2,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,...,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
3,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
4,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26281,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0
26282,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
26285,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,...,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
26286,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0


In [2]:
clf = cluster.KMeans(n_clusters=4, max_iter=300, n_init=10)
clf.fit(df_social)

KMeans(n_clusters=4)

In [3]:
df.drop(df[df['size'] <= min_size_subs].index, inplace = True)
df['cluster'] = clf.labels_

In [43]:
df_cluster = df.loc[:, ['score_plus', 'size', 'number_participants', 'subcommunities.strong', 'bottlenecks', 'number_triads', 'cluster']]
df_cluster
df_cluster.groupby(['cluster']).agg({'score_plus': 'mean', 
                                     'size': 'mean', 
                                     'number_participants': 'mean', 
                                     'subcommunities.strong': 'mean', 
                                     'bottlenecks': 'mean', 
                                     'number_triads': 'mean'})

,score_plus,size,number_participants,subcommunities.strong,bottlenecks,number_triads
cluster,,,,,,
0,56.090717,10.788125,6.449367,4.272152,1.899638,38.598855
1,42.657480,8.668080,6.403695,4.918231,2.169897,35.478498
2,128.720735,12.686875,8.121574,5.123186,1.865527,74.301838
3,72.517141,11.664821,7.318565,5.097310,2.151899,40.434599


In [45]:
coefficients = pd.read_csv('140_submission_coefficients.csv')
coefficients.drop(coefficients[np.abs(coefficients['coeff']) == 0].index, inplace = True)
features = coefficients['features']
df_cluster = df.loc[:, features]
df_cluster['score_plus'] = df['score_plus']
df_cluster['cluster'] = clf.labels_

dict_feat = {}
dict_feat['score_plus'] = 'mean'


for f in features:
    dict_feat[f] = 'mean'

df_cluster.groupby(['cluster']).agg(dict_feat).to_csv('features.csv')

,time_first_reply,duration,number_triads,depth,engagement_intensity,bottlenecks,mean_distance,width,number_triads_closed,subcommunities.strong,centr.degree,number_participants,centr.pagerank,density,score_plus,cluster
0,0.166944,0.828333,1,2,1.333333,1,1.333333,3,0,1,2.666667,3,0.333333,0.666667,5,2
1,18.610556,47.086666,0,4,2.000000,2,2.000000,1,0,1,2.000000,2,0.500000,1.000000,7,3
2,0.133056,547.708300,4014,13,1.500000,17,0.197842,109,3,106,2.671429,140,0.007143,0.009609,2179,0
3,2.578889,326.795840,3,5,2.000000,1,1.750000,4,0,2,2.800000,5,0.200000,0.350000,17,0
4,0.357222,4.006111,1,3,1.333333,1,1.000000,3,0,2,2.000000,3,0.333333,0.500000,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26281,1.718611,1.718611,2,2,1.000000,1,0.666667,3,0,3,2.000000,4,0.250000,0.333333,6,2
26282,0.042500,8.741389,21,1,0.875000,8,0.125000,7,0,8,1.750000,8,0.125000,0.125000,23,3
26285,0.123333,28.575277,1,3,1.666667,1,1.666667,3,0,1,2.666667,3,0.333333,0.666667,8,2
26286,0.071111,0.617222,1,6,2.666667,1,2.666667,2,0,1,2.666667,3,0.333333,0.666667,12,0


In [11]:
dict_feat

{'score_plus': 'mean',
 'size': 'mean',
 'number_participants': 'mean',
 'subcommunities.strong': 'mean',
 'bottlenecks': 'mean',
 'number_triads': 'mean',
 'cluster': 'mean'}

In [30]:
best_cluster = df_cluster.groupby(['cluster']).agg({'score_plus': 'mean'}).sort_values('score_plus', ascending=False).reset_index().loc[0, 'cluster']

In [42]:
df_social.columns

Index(['i', 'we', 'pronoun', 'ppron', 'ipron', 'affect', 'posemo', 'negemo',
       'work', 'power', 'drives', 'percept', 'negate', 'interrog',
       'focuspresent', 'auxverb', 'you', 'assent', 'focuspast', 'affiliation',
       'social'],
      dtype='object')

In [32]:
features = ['i', 'we', 'pronoun', 'ppron', 'ipron', 'affect', 'posemo', 'negemo', 'work', 'power', 'drives', 
         'percept', 'negate', 'interrog', 'focuspresent', 'auxverb', 'you', 'assent', 'focuspast', 
         'affiliation', 'social']
centroids = pd.DataFrame(clf.cluster_centers_, columns=features)

In [33]:
centroids

,i,we,pronoun,ppron,ipron,affect,posemo,negemo,work,power,...,percept,negate,interrog,focuspresent,auxverb,you,assent,focuspast,affiliation,social
0,0.529837,0.088909,0.979204,0.992767,0.861664,0.025919,0.107294,0.130500,0.738999,0.244424,...,0.143460,0.517480,0.649488,0.800181,0.749548,0.992767,0.062387,0.369500,0.081676,0.301386
1,0.084217,0.052408,0.161163,0.031203,0.625871,0.015753,0.083914,0.078461,0.674644,0.222054,...,0.137231,0.256286,0.312027,0.360194,0.346865,0.949106,0.066041,0.207210,0.053620,0.070585
2,0.219787,0.051563,0.655172,0.435707,0.736062,0.945537,0.987432,0.141154,0.689977,0.207219,...,0.192717,0.326781,0.382533,0.833065,0.447954,0.962617,0.247825,0.250725,0.162423,0.326458
3,0.099182,0.053020,0.706410,0.000528,0.981535,0.014508,0.063308,0.103667,0.696650,0.208652,...,0.122395,0.625956,0.754418,0.855447,0.848853,0.980744,0.057505,0.248747,0.037721,0.071749


In [34]:
best_features = []

In [35]:
for index in range(0, len(features)):
    max_value = np.argsort(np.abs(centroids.iloc[0:, index]))[3]
    if (max_value == best_cluster):
        best_features.append(centroids.columns[index])

In [38]:
K = 4
np.argsort(np.abs(centroids.iloc[0:, 0]))[K-1]

0

In [36]:
best_features

['affect',
 'posemo',
 'negemo',
 'drives',
 'percept',
 'assent',
 'affiliation',
 'social']

# Cognitive

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn import cluster
from sklearn import metrics
from scipy.spatial.distance import cdist
from yellowbrick.cluster import KElbowVisualizer

In [ ]:
min_size_subs = 3

df = pd.read_csv('100_submission_measures_binary.csv')

#feat_cognitive = ['size', 'i', 'you', 'posemo', 'negemo', 'affect', 'verb', 'focuspast', 'ppron', 
#                  'insight', 'tentat', 'pronoun', 'prep', 'adverb', 'cause', 'cogproc', 'function', 
#                  'discrep', 'certain', 'differ', 'money', 'article', 'conj', 'auxverb', 'ipron']

feat_cognitive = ['you', 'posemo', 'negemo', 'affect', 'verb', 'focuspast', 'ppron', 
                  'insight', 'tentat', 'prep', 'adverb', 'cause', 'cogproc', 'function', 
                  'discrep', 'certain', 'differ', 'article', 'conj', 'auxverb', 'ipron']

df_cognitive = df[feat_cognitive]
df_cognitive.drop(df_cognitive[df_cognitive['size'] <= min_size_subs].index, inplace = True)
df_cognitive.drop(['size'], axis=1, inplace=True)
df_cognitive

In [ ]:
df = pd.read_csv('100_submission_measures_binary.csv')

feat_social = ['i', 'we', 'pronoun', 'ppron', 'ipron', 'affect', 'posemo', 
               'negemo', 'work', 'power', 'drives', 'percept', 'negate', 'interrog', 
               'focuspresent', 'auxverb', 'you', 'assent', 'focuspast', 'affiliation', 'social']

feat_cognitive = ['you', 'posemo', 'negemo', 'affect', 'verb', 'focuspast', 'ppron', 
                  'insight', 'tentat', 'prep', 'adverb', 'cause', 'cogproc', 'function', 
                  'discrep', 'certain', 'differ', 'article', 'conj', 'auxverb', 'ipron']

liwc_features = feat_social + list(set(feat_cognitive) - set(feat_social))
print(len(liwc_features))
liwc_features

In [ ]:
clf = cluster.KMeans(n_clusters=4, max_iter=300, n_init=10)
clf.fit(df_cognitive)

In [ ]:
df.drop(df[df['size'] <= min_size_subs].index, inplace = True)
df['cluster'] = clf.labels_

In [ ]:
df_cluster = df.loc[:, ['score_plus', 'size', 'number_participants', 'subcommunities.strong', 'bottlenecks', 'number_triads', 'cluster']]
df_cluster
df_cluster.groupby(['cluster']).agg({'score_plus': 'mean', 
                                     'size': 'mean', 
                                     'number_participants': 'mean', 
                                     'subcommunities.strong': 'mean', 
                                     'bottlenecks': 'mean', 
                                     'number_triads': 'mean'})

In [ ]:
best_cluster = df_cluster.groupby(['cluster']).agg({'score_plus': 'mean'}).sort_values('score_plus', ascending=False).reset_index().loc[0, 'cluster']

In [ ]:
best_cluster

In [ ]:
features = ['you', 'posemo', 'negemo', 'affect', 'verb', 'focuspast', 'ppron', 
                  'insight', 'tentat', 'prep', 'adverb', 'cause', 'cogproc', 'function', 
                  'discrep', 'certain', 'differ', 'article', 'conj', 'auxverb', 'ipron']
centroids = pd.DataFrame(clf.cluster_centers_, columns=features)

In [ ]:
centroids

In [ ]:
best_features = []

In [ ]:
for index in range(0, len(features)):
    max_value = np.argsort(np.abs(centroids.iloc[0:, index]))[3]
    if (max_value == best_cluster):
        best_features.append(centroids.columns[index])

In [ ]:
best_features